# Manual clean business rebuild — DEV then PROD

Read `docs/privacy_rebuild.md` first. This is maintenance, NOT a scheduled Job.
Default `plan` reads Spark and saves a private checkpoint only. Run one stage at a time:
`plan` → `reset` → `monthly` → `daily` → `validate`.
RESET also removes the known legacy charge_subcategory from the two empty charge tables.
For a diagnosed first-month failure only, see `recover_charge_schema` in the maintenance guide.

No catalog/schema/Volume/GCS deletion. Only the selected environment's 30 managed
business tables are truncated after explicit confirmation. OPS history and
entitlements are retained; normal reloads add new audit events. Historical Delta
files are not erased. Do not publish old recovery output or private checkpoints.


In [ ]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / 'src/finops_cloud').is_dir()), None)
if PROJECT_ROOT is None:
    PROJECT_ROOT = Path('/Workspace/Users/thailongdam@gmail.com/finops-cloud_data_platform')
if not (PROJECT_ROOT / 'src/finops_cloud').is_dir():
    raise ValueError('Set PROJECT_ROOT to the Belgian Databricks Git Folder')
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from finops_cloud.config import load_config
from finops_cloud.medallion.privacy_rebuild import run_stage


## Parameters — review before EVERY stage

Create the private Workspace folder below yourself, outside Git; restrict its access.
DEV and PROD use separate checkpoint files. Never delete/edit a checkpoint to bypass a
failure. Leave confirmations empty/False for PLAN. Before writing, confirm the real
jobs and dashboard are paused and the source publication is verified.

A missing checkpoint, unexpected table version, wrong metastore, interrupted operation
or failed stage blocks continuation. Do not run the old catalog-drop or text-mapping
repair procedure, and do not launch the normal orchestration Job during maintenance.


In [ ]:
ENVIRONMENT = 'dev'
STAGE = 'plan'
START_MONTH = '2025-01'
END_MONTH = '2026-06'
CONTROL_DIRECTORY = Path('/Workspace/Users/thailongdam@gmail.com/privacy-rebuild-20261006')

CONFIRMATION = ''  # Normal writes: REBUILD_DEV/PROD_BUSINESS_DATA; recovery: see docs/privacy_rebuild.md.
JOBS_PAUSED = False
DASHBOARD_PAUSED = False
SOURCES_VERIFIED = False

# Plain notebook variables: no stale Job widgets override the maintenance choice.
config = load_config(ENVIRONMENT, PROJECT_ROOT)
state_file = CONTROL_DIRECTORY / f'rebuild_{ENVIRONMENT}.json'
print('Review:', {'environment': ENVIRONMENT, 'catalog': config.catalog,
                  'stage': STAGE, 'checkpoint': str(state_file)})


In [ ]:
result = run_stage(
    spark, config, state_file,
    stage=STAGE, start_month=START_MONTH, end_month=END_MONTH,
    confirmation=CONFIRMATION, jobs_paused=JOBS_PAUSED,
    dashboard_paused=DASHBOARD_PAUSED, sources_verified=SOURCES_VERIFIED,
)
print(json.dumps(result, indent=2))
if STAGE == 'plan':
    control = json.loads(state_file.read_text())
    print('RECOVERY VERSIONS — save privately:', json.dumps(control['recovery_versions']))
    print('Active sources — review privately:', json.dumps(control['sources'], indent=2))


## After DEV/PROD PASS

DEV PASS does not authorize PROD automatically: repeat PLAN with `ENVIRONMENT='prod'`
and preserve that environment's own baseline. Before resetting PROD, actually suspend
public dashboard reads; closing one browser tab is insufficient.

After BOTH environments pass, execute security SQL 04 then 05 in PROD and test Owner A,
Owner B, Admin and No Access. Only then approve the public `portfolio_demo` deployment.
Its Admin must not receive raw OPS histories. Keep past source files, Delta snapshots,
operator logs and recovery records private; this procedure does not prove complete
anonymity or erase retention history. On failure keep jobs/dashboard paused and save
all output: no automatic or one-table-only RESTORE is performed.
